In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, privacy

results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print("PASS" if ok else "FAIL", name, detail)

In [0]:
import re

p = privacy.load()
READ = {"READ VOLUME", "WRITE VOLUME", "ALL PRIVILEGES"}      # each of these lets a principal read the files
VOLUMES = ["landing", "processed", "rejected"]
UUID = r"^[0-9a-f]{8}-([0-9a-f]{4}-){3}[0-9a-f]{12}$"

def label(principal):                       # never print a person's login
    return "<a user>" if "@" in principal else principal

def readers(cat, volume):
    """Everyone who can read the volume: grants on the volume itself, its schema and its catalog."""
    found = set()
    for kind, name in (("VOLUME", f"{cat}.bronze.{volume}"), ("SCHEMA", f"{cat}.bronze"), ("CATALOG", cat)):
        for r in spark.sql(f"SHOW GRANTS ON {kind} {name}").collect():
            if r[1].replace("_", " ").upper() in READ:
                found.add(r[0])
    return found

for e in ["dev", "test", "prod"]:
    cat = common.load_config(e)["catalog"]
    allowed = set(p["clear"][e])
    for v in VOLUMES:
        name = f"{cat}.bronze.{v}: only clear groups and the job account can read raw files"
        try:
            seen = readers(cat, v)
        except Exception as err:
            msg = str(err).split("\n")[0]
            if "PERMISSION_DENIED" in msg and "READ VOLUME" in msg:
                check(name, True, "this login is refused (no READ VOLUME)")      # being refused is the proof
            else:
                check(name, False, msg[:140])
            continue
        jobs = {x for x in seen if re.match(UUID, x)} if e in p.get("pipeline", {}) else set()
        extra = sorted(label(x) for x in seen - allowed - jobs)
        print(f"INFO {cat}.bronze.{v}: readers {sorted(label(x) for x in seen)}")
        check(name, not extra, str(extra))

print(f"\nt16_volume_access: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t16_volume_access failed")